# FDA EDA (2019-2026): ACT charge mapping + SPS/TBT categorization

This notebook:
1. Maps `REFUSAL_ENTRY` records to ACT charge master (`ACT_SECTION_CHARGES`) by `REFUSAL_CHARGES -> ASC_ID`.
2. Filters to the same subjects: Vietnam (`VN`), India (`IN`), Ecuador (`EC`).
3. Classifies refusals into `SPS`, `TBT`, `SPS+TBT`, or `Other/Unmatched` by keyword rules on ACT charge text.
4. Runs period-wise analysis (`2019-2023`, `2024-2026`) and concatenates final output for `2019-2026`.

In [1]:
import re
from pathlib import Path
import pandas as pd

# Same subjects as EU analysis
TARGET_COUNTRY_CODES = {"VN", "IN", "EC"}
COUNTRY_NAME_MAP = {"VN": "Vietnam", "IN": "India", "EC": "Ecuador"}
START_DATE = pd.Timestamp("2020-03-30")
END_DATE = pd.Timestamp("2025-12-31")

SPS_KEYWORDS = [
    r"\bsps\b",
    r"sanitary",
    r"phytosanitary",
    r"pesticide",
    r"residue",
    r"mr\s*l",
    r"microbiolog",
    r"salmonella",
    r"listeria",
    r"aflatoxin",
    r"mycotoxin",
    r"contamin",
    r"pathogen",
    r"veterinary",
    r"animal health",
    r"plant health",
    r"food safety",
    r"filth",
    r"decompos",
    r"adulterat",
]

TBT_KEYWORDS = [
    r"\btbt\b",
    r"technical barrier",
    r"technical regulation",
    r"misbrand",
    r"label",
    r"labelling",
    r"packag",
    r"certificate",
    r"documentation",
    r"traceability",
    r"standard",
    r"marking",
    r"conformity",
    r"compliance",
    r"declaration",
    r"statement",
    r"identity",
]

SPS_PATTERN = re.compile("|".join(SPS_KEYWORDS), flags=re.IGNORECASE)
TBT_PATTERN = re.compile("|".join(TBT_KEYWORDS), flags=re.IGNORECASE)

BASE_DIR = Path("")

In [2]:
def read_csv_robust(file_path: Path) -> pd.DataFrame:
    encodings = ["utf-8", "utf-8-sig", "cp1252", "latin1"]
    for enc in encodings:
        try:
            return pd.read_csv(file_path, encoding=enc, low_memory=False)
        except UnicodeDecodeError:
            continue
        except pd.errors.ParserError:
            try:
                return pd.read_csv(
                    file_path,
                    encoding=enc,
                    low_memory=False,
                    engine="python",
                    on_bad_lines="skip",
                )
            except Exception:
                continue

    # Last fallback keeps pipeline running on malformed bytes/rows.
    return pd.read_csv(
        file_path,
        encoding="utf-8",
        encoding_errors="replace",
        low_memory=False,
        engine="python",
        on_bad_lines="skip",
    )


def load_period_data(refusal_file: Path, act_file: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    refusal_df = read_csv_robust(refusal_file)
    act_df = read_csv_robust(act_file)

    act_df["ASC_ID"] = pd.to_numeric(act_df["ASC_ID"], errors="coerce").astype("Int64")
    refusal_df["ISO_CNTRY_CODE"] = refusal_df["ISO_CNTRY_CODE"].astype(str).str.upper().str.strip()

    refusal_df["REFUSAL_DATE_PARSED"] = pd.to_datetime(
        refusal_df["REFUSAL_DATE"],
        errors="coerce",
        format="mixed",
        utc=False,
    )

    country_mask = refusal_df["ISO_CNTRY_CODE"].isin(TARGET_COUNTRY_CODES)
    date_mask = refusal_df["REFUSAL_DATE_PARSED"].between(START_DATE, END_DATE, inclusive="both")
    refusal_df = refusal_df[country_mask & date_mask].copy()

    # Stable row id so each refusal row can be re-aggregated after exploding charges.
    refusal_df = refusal_df.reset_index(drop=True)
    refusal_df["refusal_row_id"] = refusal_df.index

    return refusal_df, act_df


def explode_refusal_charges(refusal_df: pd.DataFrame) -> pd.DataFrame:
    # REFUSAL_CHARGES values are like: "11, 321, 482".
    charge_series = refusal_df["REFUSAL_CHARGES"].fillna("").astype(str)
    extracted = charge_series.str.findall(r"\d+")

    exploded = refusal_df[["refusal_row_id"]].copy()
    exploded["ASC_ID"] = extracted
    exploded = exploded.explode("ASC_ID")
    exploded = exploded.dropna(subset=["ASC_ID"]).copy()
    exploded["ASC_ID"] = pd.to_numeric(exploded["ASC_ID"], errors="coerce").astype("Int64")
    exploded = exploded.dropna(subset=["ASC_ID"])

    return exploded


def classify_charge_text(text: str) -> str:
    txt = str(text)
    has_sps = bool(SPS_PATTERN.search(txt))
    has_tbt = bool(TBT_PATTERN.search(txt))

    if has_sps and has_tbt:
        return "SPS+TBT"
    if has_sps:
        return "SPS"
    if has_tbt:
        return "TBT"
    return "Other/Unmatched"


def process_period(refusal_file: Path, act_file: Path, period_label: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    refusal_df, act_df = load_period_data(refusal_file, act_file)
    exploded = explode_refusal_charges(refusal_df)

    merged = exploded.merge(
        act_df[["ASC_ID", "CHRG_CODE", "CHRG_STMNT_TEXT", "SCTN_NAME"]],
        on="ASC_ID",
        how="left",
    )

    merged["charge_text"] = (
        merged["CHRG_CODE"].fillna("").astype(str)
        + " | " + merged["CHRG_STMNT_TEXT"].fillna("").astype(str)
        + " | " + merged["SCTN_NAME"].fillna("").astype(str)
    )
    merged["charge_category"] = merged["charge_text"].apply(classify_charge_text)

    # Aggregate from charge-level to refusal-level classification.
    per_row = merged.groupby("refusal_row_id", as_index=False).agg(
        mapped_charge_count=("ASC_ID", "nunique"),
        mapped_charge_categories=("charge_category", lambda s: ", ".join(sorted(set(s.dropna().astype(str))))),
        mapped_charge_codes=("CHRG_CODE", lambda s: ", ".join(sorted(set(v for v in s.dropna().astype(str) if v.strip())))),
    )

    out = refusal_df.merge(per_row, on="refusal_row_id", how="left")
    out["mapped_charge_count"] = out["mapped_charge_count"].fillna(0).astype(int)
    out["mapped_charge_categories"] = out["mapped_charge_categories"].fillna("Other/Unmatched")
    out["mapped_charge_codes"] = out["mapped_charge_codes"].fillna("")

    def classify_refusal(categories: str) -> str:
        cats = set([c.strip() for c in str(categories).split(",") if c.strip()])
        has_sps = any(c in {"SPS", "SPS+TBT"} for c in cats)
        has_tbt = any(c in {"TBT", "SPS+TBT"} for c in cats)
        if has_sps and has_tbt:
            return "SPS+TBT"
        if has_sps:
            return "SPS"
        if has_tbt:
            return "TBT"
        return "Other/Unmatched"

    out["sps_tbt_category"] = out["mapped_charge_categories"].apply(classify_refusal)
    out["country_name"] = out["ISO_CNTRY_CODE"].map(COUNTRY_NAME_MAP).fillna(out["ISO_CNTRY_CODE"])
    out["period"] = period_label

    summary = (
        out.groupby(["period", "ISO_CNTRY_CODE", "country_name", "sps_tbt_category"], as_index=False)
        .size()
        .rename(columns={"size": "refusal_count"})
        .sort_values(["country_name", "sps_tbt_category"])
)

    return out, summary

In [3]:
print(f"Validation date window: {START_DATE.date()} to {END_DATE.date()}")

# Period-specific processing
period1_out, period1_summary = process_period(
    BASE_DIR / "REFUSAL_ENTRY_2019_2023.csv",
    BASE_DIR / "ACT_SECTION_CHARGES_1923.csv",
    "2019-2023",
)

period2_out, period2_summary = process_period(
    BASE_DIR / "REFUSAL_ENTRY_2024-Feb2026.csv",
    BASE_DIR / "ACT_SECTION_CHARGES_2426.csv",
    "2024-2026",
)

# Concatenate for full-period output
full_out = pd.concat([period1_out, period2_out], ignore_index=True)
full_summary = (
    full_out.groupby(["ISO_CNTRY_CODE", "country_name", "sps_tbt_category"], as_index=False)
    .size()
    .rename(columns={"size": "refusal_count"})
    .sort_values(["country_name", "sps_tbt_category"])
)

# Save outputs
period1_path = BASE_DIR / "FDA_2019_2023_mapped_sps_tbt.csv"
period2_path = BASE_DIR / "FDA_2024_2026_mapped_sps_tbt.csv"
full_path = BASE_DIR / "FDA_2019_2026_mapped_sps_tbt.csv"
summary_period_path = BASE_DIR / "FDA_2019_2026_period_summary.csv"
summary_full_path = BASE_DIR / "FDA_2019_2026_country_category_summary.csv"

period1_out.to_csv(period1_path, index=False, encoding="utf-8-sig")
period2_out.to_csv(period2_path, index=False, encoding="utf-8-sig")
full_out.to_csv(full_path, index=False, encoding="utf-8-sig")
pd.concat([period1_summary, period2_summary], ignore_index=True).to_csv(summary_period_path, index=False, encoding="utf-8-sig")
full_summary.to_csv(summary_full_path, index=False, encoding="utf-8-sig")

print("Saved files:")
print(period1_path.resolve())
print(period2_path.resolve())
print(full_path.resolve())
print(summary_period_path.resolve())
print(summary_full_path.resolve())

print("\nRows by period:")
print(full_out["period"].value_counts().to_string())

print("\nMin/Max parsed refusal date in output:")
print(full_out["REFUSAL_DATE_PARSED"].agg(["min", "max"]).to_string())

print("\nFull-period country x category summary:")
full_summary

Validation date window: 2020-03-30 to 2025-12-31
Saved files:
D:\Phuong\Code in Python\KTPT\FDA\FDA_2019_2023_mapped_sps_tbt.csv
D:\Phuong\Code in Python\KTPT\FDA\FDA_2024_2026_mapped_sps_tbt.csv
D:\Phuong\Code in Python\KTPT\FDA\FDA_2019_2026_mapped_sps_tbt.csv
D:\Phuong\Code in Python\KTPT\FDA\FDA_2019_2026_period_summary.csv
D:\Phuong\Code in Python\KTPT\FDA\FDA_2019_2026_country_category_summary.csv

Rows by period:
period
2024-2026    11021
2019-2023     9163

Min/Max parsed refusal date in output:
min   2020-03-30
max   2025-12-31

Full-period country x category summary:


,ISO_CNTRY_CODE,country_name,sps_tbt_category,refusal_count
0,EC,Ecuador,Other/Unmatched,54
1,EC,Ecuador,SPS,181
2,EC,Ecuador,SPS+TBT,19
3,EC,Ecuador,TBT,98
4,IN,India,Other/Unmatched,7232
5,IN,India,SPS,6816
6,IN,India,SPS+TBT,897
7,IN,India,TBT,3277
8,VN,Vietnam,Other/Unmatched,59
9,VN,Vietnam,SPS,1168
